# Библиотеки

In [ ]:
!pip install -q accelerate==0.33.0 langchain_huggingface bitsandbytes==0.42.0 chromadb==0.5.5 gensim==4.3.2 langchain==0.2.5 langchain-community==0.2.5 matplotlib==3.6.2 nltk==3.8.1 numpy==1.26.4 pandas==2.0.3 peft==0.11.1 scikit-learn==1.3.2 scipy==1.10.1 sentence-transformers==3.0.1 seqeval==1.2.2 tokenizers==0.19.1 torch==2.3.1 torchvision==0.18.1 transformers==4.44.0 wandb==0.13.10 autoawq==0.2.6 autoawq

In [ ]:
!pip install langchain-huggingface

In [ ]:
!pip install --upgrade torch torchvision

In [ ]:
!pip install langchain_chroma

In [ ]:
!pip install -U langchain-community

In [ ]:
!pip install autoawq

In [ ]:
!pip install protobuf==4.49.0

In [ ]:
!pip install intel_extension_for_pytorch

In [ ]:
!pip install --upgrade autoawq

In [ ]:
!pip install ddgs

# База знаний

База знаний состоит из первых абзацев русскоязычных статей из википедии про различных людей.

In [ ]:
import ipywidgets as widgets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import json

with open('/content/drive/My Drive/Colab Notebooks/DL/Трансформеры/LLM/Project/ru_wiki_person.txt', 'r', encoding="utf_8_sig") as f:
    articles = f.read().split('\n\n')

len(articles)

269086

In [ ]:
articles[:5]

['Эльда́р Алекса́ндрович Ряза́нов (18 ноября 1927, Самара, СССР — 30 ноября 2015, Москва, Россия) — советский и российский кинорежиссёр, сценарист, актёр, поэт, драматург, телеведущий, педагог, продюсер; народный артист СССР (1984), лауреат Государственной премии СССР (1977) и Государственной премии РСФСР имени братьев Васильевых (1979).Среди шедевров советской киноклассики, созданных Эльдаром Рязановым, — комедии и мелодрамы «Карнавальная ночь» (1956), «Девушка без адреса» (1957), «Дайте жалобную книгу» (1965), «Берегись автомобиля» (1966), «Старики-разбойники» (1971), «Невероятные приключения итальянцев в России» (1973), «Ирония судьбы, или С лёгким паром» (1976), «Служебный роман» (1977), «Гараж» (1979), «О бедном гусаре замолвите слово» (1980), «Вокзал для двоих» (1982), «Жестокий романс» (1984), «Небеса обетованные» (1991).Рязанов — автор более 200 собственных телевизионных программ, с 1979 по 1985 год вёл телепередачу «Кинопанорама». Автор текста ряда широко популярных романсов, 

In [ ]:
from langchain_chroma import Chroma

In [ ]:
import shutil
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain_core.documents.base import Document

embedding_model = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-large")
docs = [Document(page_content=article) for article in articles[:100]]

db = Chroma.from_documents(
    docs,
    embedding_model,
    persist_directory="chroma_db",
)

shutil.make_archive('chroma_db', 'zip', 'chroma_db')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


'/content/chroma_db.zip'

In [ ]:
import zipfile

with zipfile.ZipFile('/content/chroma_db.zip', 'r') as zip_ref:
    zip_ref.extractall('./')

chroma_client = Chroma(persist_directory="chroma_db", embedding_function=embed_model)

In [ ]:
# # разархивация

# import shutil
# shutil.unpack_archive("chroma_db.zip", "chroma_db")

# Retrieval Augmented Generation

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import AwqConfig
import torch

# Инициализация модели с конфигурацией AWQ
quantization_config = AwqConfig(
    bits=4,  # 4-битное квантование
    group_size=128,  # Размер группы для квантования
    zero_point=True,  # Использование нулевой точки
    version="gemm"  # Версия AWQ
)

# Загрузка модели и токенизатора
model_name = "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,  # Использование float16 для экономии памяти
    device_map={"": "cuda:0"},
    quantization_config=quantization_config,  # Конфигурация AWQ
    low_cpu_mem_usage=True
)

if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
pad_token_id = tokenizer.convert_tokens_to_ids('[PAD]')

`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.However, loading attributes (e.g. ['version', 'fuse_max_seq_len', 'exllama_config', 'modules_to_fuse', 'do_fuse']) will be overwritten with the one you passed to `from_pretrained`. The rest will be ignored.
  warnings.warn(warning_msg)
/usr/local/lib/python3.12/dist-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the T

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
with open("/content/drive/My Drive/Colab Notebooks/DL/Трансформеры/karpov/LLM/Project/questions.txt", "r", encoding="utf_8_sig") as f:
    questions = f.read().splitlines()

In [ ]:
model.to("cuda:0")

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): WQLinear_GEMM(in_features=4096, out_features=4096, bias=False, w_bit=4, group_size=128)
          (k_proj): WQLinear_GEMM(in_features=4096, out_features=1024, bias=False, w_bit=4, group_size=128)
          (v_proj): WQLinear_GEMM(in_features=4096, out_features=1024, bias=False, w_bit=4, group_size=128)
          (o_proj): WQLinear_GEMM(in_features=4096, out_features=4096, bias=False, w_bit=4, group_size=128)
        )
        (mlp): LlamaMLP(
          (gate_proj): WQLinear_GEMM(in_features=4096, out_features=14336, bias=False, w_bit=4, group_size=128)
          (up_proj): WQLinear_GEMM(in_features=4096, out_features=14336, bias=False, w_bit=4, group_size=128)
          (down_proj): WQLinear_GEMM(in_features=14336, out_features=4096, bias=False, w_bit=4, group_size=128)
          (ac

In [ ]:
import torch

@torch.no_grad()
def generate_answer(question, context):
    system_message = (
        "Ты полезный ассистент.\n"
        "Пожалуйста, дай ответ на запрос пользователя, используя только данную тебе информацию в контексте.\n"
        "Убедись, что твой ответ точен, короток и не содержит никакой другой информации.\n"
        "ответ не должен содержать более 7 слов, ответ должен быть уверенным и на русском языке.\n"
        f"Контекст: {context}\n"
    )
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": f"Запрос: {question}"}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False)


    tokenize_prompt = tokenizer(
        prompt,
        return_tensors="pt",
        padding=True,
        truncation=True,
        return_attention_mask=True
    )
    tokenize_prompt = {k: v.to("cuda:0") for k, v in tokenize_prompt.items()}

    # Генерация ответа
    outputs = model.generate(
        **tokenize_prompt,
        min_length=20,
        max_new_tokens=512,
        do_sample=True,
        top_k=5,
        temperature=0.3,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=pad_token_id,
        repetition_penalty=1.2,  # Исправлено: repetition_penalty как число
    )

    decode_outputs = tokenizer.decode(outputs[0], skip_special_tokens=True).replace("\n", "").strip()
    if "assistant:" in decode_outputs:
        decode_outputs = decode_outputs.split("assistant:")[-1].strip()
    if "assistant" in decode_outputs:
        decode_outputs = decode_outputs.split("assistant")[-1].strip()
    if "Ответ:" in decode_outputs:
        decode_outputs = decode_outputs.split("Ответ:")[-1].strip().split(". ")[0]
    if "ответ:" in decode_outputs:
        decode_outputs = decode_outputs.split("ответ:")[-1].strip().split(". ")[0]
    return decode_outputs

In [ ]:
generated_answers = []
for question in ['Кто сыграл Росомаху в серии фильмов "Люди Икс"?']:
    similarity_texts = chroma_client.similarity_search_with_relevance_scores(
        query=question,
        k=3,
    )

    print(similarity_texts)
    context = "\n".join([text.page_content for text, _ in similarity_texts])
    print(context)
    answer = generate_answer(question, context)
    print(answer)
    generated_answers.append(answer)
    break

[(Document(id='b1e42854-385f-442e-8975-a39b4dca1e6b', metadata={}, page_content='Сильве́стр Гарде́нцио Сталло́не ( ; ; род. 6 июля 1946, Нью-Йорк, США) — американский актёр, кинорежиссёр, сценарист и продюсер. За свою актёрскую карьеру снялся более чем в 50 фильмах, в том числе в сериях фильмов «Рокки» (1976—2018), «Рэмбо» (1982—2019), «Неудержимые» (2010 — наст. время) и «План побега» (2013). К первым трём из которых также приложил руку (и не одну) в качестве сценариста, режиссёра и продюсера.Номинант на премию «Золотой глобус» за роль Рокки Бальбоа в первом фильме и лауреат за роль в . Трёхкратный номинант на премию «Оскар» за роль Рокки Бальбоа в первом и седьмом фильмах, а также за сценарий к первому фильму, лауреат премий «Сатурн», «Сезар» и «Critics’ Choice Movie Awards».По результатам на 2012 год общая касса фильмов со Сталлоне в качестве актёра составила 3,7 миллиарда долларов (с поправкой на инфляцию).12 июня 2011 года за заслуги в популяризации бокса был введён в Международны

In [ ]:
# Цикл для обработки вопросов
generated_answers = []
for question in questions:
    answer = generate_answer(question, 3)
    generated_answers.append(answer)

In [ ]:
generated_answers

['Николай Аммосов.',
 'Теория относительности.',
 'Джордж Оруэлл.',
 'Невилл Чемберлен.',
 'Майкл Джексон.',
 'Леонардо да Винчи.',
 'Стэнфорд Каррерас.',
 'Абрахам Линкольн.',
 'Винцент ван Гог.',
 'Вильям Шекспир.',
 'Хью Джекман.',
 'Наполеон Бонапарт.',
 'Бизе.',
 'Густав Эйфель.',
 'Иоосиф Сталин.',
 'Альберт Эйнштейн.',
 'Александр Грамм.',
 'Лев Толстой.',
 'Леонардо да Винчи.',
 'Билл Гейтс.',
 'Хэлли Берри.',
 'Юрий Гагарин.',
 'Лудвиг ван Бетховен.',
 'Христофор Колумб.',
 'Сигмунд Фрейд.',
 'Это был Юрий Гагарин.',
 'Эрнест Хемингуэй. Нет, это неверно. Правильный автор - Герман Мелвилл.',
 'Хуан Грис.',
 'Джордж Вашингтон.',
 'Вильям Шекспир.',
 'Элон Маск.',
 'Моцарт.',
 'Дэниел Рэдклифф.',
 'Фёдор Достоевский.',
 'Джавахарлал Неру.',
 'Михаил Ангелович Микеланджело.',
 'Вильям Шекспир.',
 'Чарльз Линдберgh.',
 'Марк Цукерберг.',
 'Менделье.',
 'Мидж Марвел Хоппер.',
 'Роберт Дауни- мл.',
 'Первый человек - Тэнзин Норге.',
 'Габриэль Гарсия Маркес.',
 'Хардинг.',
 'Ричард Б

In [ ]:
# with open("/content/drive/My Drive/Colab Notebooks/Karpov_curse/answers_new.json", "w", encoding="utf_8") as f:
#     json.dump(generated_answers, f, ensure_ascii=False)

# Поиск в интернете



In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

In [ ]:
import re
import requests
import ddgs
from bs4 import BeautifulSoup
from ddgs import DDGS

In [ ]:
# код с обратоткой ошибок
import requests
from bs4 import BeautifulSoup
import re
import socket
from urllib3.exceptions import NameResolutionError, MaxRetryError
from requests.exceptions import ConnectionError, RequestException

def safe_internet_search(query, k):
    ddgs = DDGS()
    try:
        result = ddgs.text(query, max_results=k)
        links = [x['href'] for x in result]
    except Exception as e:
        print(f"Ошибка при поиске по DDGS: {e}")
        return []

    headers = {
        "User-Agent": 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                      'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/108.0.0.0 Safari/537.36'
    }

    out_texts = []
    for link in links:
        link = link.replace('%25', '%').replace('%3F', '?').replace('%3D', '=')
        try:
            response = requests.get(link, headers=headers, timeout=5)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, "html.parser")
            text = re.sub(r'[\n]+', '\n', soup.get_text()).replace('\xa0', ' ')
            out_texts.append(text)
        except (socket.gaierror, NameResolutionError, MaxRetryError, ConnectionError, RequestException) as e:
            print(f"Ошибка при получении страницы {link}: {e}. Пропускаю.")
            continue
        except Exception as e:
            print(f"Неизвестная ошибка при обработке {link}: {e}. Пропускаю.")
            continue

    return out_texts


# Поддержка диалогов

 Для того, чтобы по новому вопросу можно было достать релевантные тексты из базы данных, вопрос нужно переформулировать, добавив нужную информацию из предыдущих сообщений пользователя. Поэтому при получении нового вопроса можно сделать запрос в LLM для уточнения запроса пользователя с учетом всей истории сообщений, а после этого искать релевантные тексты по уточненному запросу.

In [ ]:
import numpy as np

In [ ]:
def reformulate_question(question, history, n=1):
    messages = [
        {"role": "system", "content": f'''Твоя задача написать {n} вариаций вопроса пользователя для того,
чтобы по ним получить релевантные документы из векторной базы данных.
Напиши ТОЛЬКО вариации вопроса и больше ничего, разделяя их символом новой строки \\\\n.
НЕ пиши ответ на сам вопрос."'''},
        {"role": "user", "content": f"История диалога: {history}\nВопрос: {question}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False)
    tokenized_prompt = tokenizer(prompt, padding=True, return_attention_mask=True, return_tensors="pt")
    tokenized_prompt = {k: v.to("cuda:0") for k, v in tokenized_prompt.items()}
    outputs = model.generate(
        **tokenized_prompt,
        max_new_tokens=256,
        do_sample=True,
        top_k=50,
        top_p=0.9,
        temperature=0.3,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.2
    )

    decode_outputs = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    if "assistant:" in decode_outputs:
        decode_outputs = decode_outputs.split("assistant:")[-1].strip()
    if "Ответ:" in decode_outputs:
        decode_outputs = decode_outputs.split("Ответ:")[-1].strip().split(". ")[0]
    if "ответ:" in decode_outputs:
        decode_outputs = decode_outputs.split("ответ:")[-1].strip().split(". ")[0]

    decode_outputs = decode_outputs.split("\\\\n\n")[: n]

    return decode_outputs

In [ ]:
reformulate_question('Сколько фильмов о Людях Икс с этим актёром было выпущено?', 'user: Кто сыграл Росомаху в серии фильмов "Люди Икс"?\nassistant: Хью Джекман.', n=5)

['\\\\н \nКоличество фильмов о Людях Икс с участием Хью Джекмана.',
 'Список всех фильмов о Людях Икс с ролью Росомахи.',
 'Фильмы о Людях Икс с исполнением Росомахи актером Хью Джекманом.',
 'Какое количество картин о Людях Икс включает в себя персонажа Росомахи исполняемого Хью Джекманом.',
 'Число фильмов про Людей Икс где играл Росомаху Хью Джекман.']

In [ ]:
generated_answers = []
for question in ['Кто написал пьесу "Ромео и Джульетта"?']:
    similarity_texts = chroma_client.similarity_search(
        query=question,
        k=3,
    )
    # print(similarity_texts)
    context = "\n".join([text.page_content for text in similarity_texts])
    print(context)
    answer = generate_answer(question, context)
    print(answer)
    generated_answers.append(answer)
    break

In [ ]:
generated_answers

['Шекспир.']

In [ ]:
with open("/content/drive/My Drive/Colab Notebooks/DL/Трансформеры/LLM/Project/dialog_questions.txt", "r") as f:
    questions = f.read().splitlines()

In [ ]:
questions = [question for question in questions if question != ""]

In [ ]:
questions

['Кто первым человеком высадился на Луну?',
 'В каком году состоялась эта высадка?',
 'Кто написал "Божественную комедию"?',
 'На каком языке написана эта поэма?',
 'Кто написал роман "1984"?',
 'В каком году был опубликован этот роман?',
 'Кто был премьер-министром Великобритании во время Второй мировой войны?',
 'Сколько лет он находился на посту премьер-министра?',
 'Кто исполнил песню "Thriller"?',
 'В каком году был выпущен альбом с этой песней?',
 'Кто создал картину "Мона Лиза"?',
 'В каком музее находится эта картина?',
 'Кто основал компанию Apple?',
 'В каком году она была основана?',
 'Кто был президентом США, подписавшим Прокламацию об освобождении рабов?',
 'В каком году это произошло?',
 'Кто нарисовал "Звёздную ночь"?',
 'В каком музее выставлена эта картина?',
 'Кто написал пьесу "Ромео и Джульетта"?',
 'В каком городе происходит действие этой пьесы?',
 'Кто сыграл Росомаху в серии фильмов "Люди Икс"?',
 'Сколько фильмов о Людях Икс с этим актёром было выпущено?',
 'Кто

In [ ]:
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


In [ ]:
import json
import sys
import numpy as np
import torch
import gc

In [ ]:
def process_questions(questions, similarity_threshold=0.8, k=3, max_context_length=3000):
    answers, history = [], ""
    for i in range(0, len(questions), 2):
        answer_pair = []
        first_question, second_question = questions[i], questions[i + 1]
        print(first_question, second_question)

        similarity_texts_with_scores = chroma_client.similarity_search_with_relevance_scores(
            query=first_question,
            k=k,
        )
        filtered_pairs = [(text, score) for text, score in similarity_texts_with_scores
                          if score >= similarity_threshold]

        if len(filtered_pairs) < k:
            print(f"Недостаточно релевантных документов в базе. Провожу интернет-поиск по запросу: {first_question}")
            search_results = safe_internet_search(first_question, k=k - len(filtered_pairs))
            combined_texts = [text.page_content for text, _ in filtered_pairs] + search_results
            context = "\n".join(combined_texts)
            del search_results
            if len(context) > max_context_length:
                context = context[:max_context_length]
        else:
            context = "\n".join([text.page_content for text, _ in filtered_pairs])
            if len(context) > max_context_length:
                context = context[:max_context_length]

        answer = generate_answer(first_question, context)
        answer_pair.append(answer)
        history += f"user: {first_question}\nassistant: {answer}\n"

        reformulated_questions = reformulate_question(second_question, history, n=3)

        candidates_for_context = []
        for reformulated_question in reformulated_questions:
            similarity_texts_with_scores = chroma_client.similarity_search_with_relevance_scores(
                reformulated_question,
                k=k,
            )
            filtered_pairs_sub = [(text, score) for text, score in similarity_texts_with_scores
                                  if score >= similarity_threshold]
            candidates_for_context.extend([text.page_content for text, _ in filtered_pairs_sub])

        candidates_for_context = np.unique(candidates_for_context).tolist()

        if len(candidates_for_context) < k:
            print(f"Недостаточно релевантных документов в базе. Провожу интернет-поиск по запросу: {second_question}")
            search_results = safe_internet_search(second_question, k=k - len(candidates_for_context))
            combined_texts = list(candidates_for_context) + list(search_results)
            context = "\n".join(combined_texts)
            del search_results
            if len(context) > max_context_length:
                context = context[:max_context_length]
        else:
            context = "\n".join(candidates_for_context)
            if len(context) > max_context_length:
                context = context[:max_context_length]

        answer = generate_answer(f"История диалога: {history}\nВопрос: {second_question}", context)
        answer_pair.append(answer)
        answers.append(answer_pair)
        print(answer_pair)

        # Очищаем историю и контекст, а также кэш
        history = ""
        context = ""
        del candidates_for_context, reformulated_questions, filtered_pairs_sub

        torch.cuda.empty_cache()
        gc.collect()

    return answers


In [ ]:
def chunk_list(lst, chunk_size):
    return [lst[i:i + chunk_size] for i in range(0, len(lst), chunk_size)]

answers = []
chunked_questions = chunk_list(questions, 6)

for questions_chunk in chunked_questions:
    partial_answers = process_questions(questions_chunk)
    answers.extend(partial_answers)


Кто первым человеком высадился на Луну? В каком году состоялась эта высадка?
Недостаточно релевантных документов в базе. Провожу интернет-поиск по запросу: Кто первым человеком высадился на Луну?
['Нил Армстронг.', '1969 год.']
Кто написал "Божественную комедию"? На каком языке написана эта поэма?
Недостаточно релевантных документов в базе. Провожу интернет-поиск по запросу: Кто написал "Божественную комедию"?
['Данте Алигьери.', 'итальянский язык.']
Кто написал роман "1984"? В каком году был опубликован этот роман?
Недостаточно релевантных документов в базе. Провожу интернет-поиск по запросу: Кто написал роман "1984"?
Ошибка при получении страницы https://medium.com/kkblog/джордж-оруэлл-1984-37e34e14be62: 429 Client Error: Too Many Requests for url: https://medium.com/kkblog/%D0%B4%D0%B6%D0%BE%D1%80%D0%B4%D0%B6-%D0%BE%D1%80%D1%83%D1%8D%D0%BB%D0%BB-1984-37e34e14be62. Пропускаю.
['Джордж Оруэлл.', '1949 год.']
Кто был премьер-министром Великобритании во время Второй мировой войны? Сколь

In [ ]:
answers

[['Нил Армстронг.', '1969 год.'],
 ['Данте Алигьери.', 'итальянский язык.'],
 ['Джордж Оруэлл.', '1949 год.'],
 ['Вinston Churchill.', '21 год'],
 ['Майкл Джексон.', '1982 год.'],
 ['Леонардо да Винчи.', 'Лувр.'],
 ['Стив Возняк, Рональд Уэйн и Стив Джобс.', '1976 год.'],
 ['Авраам Линкольн.', '1862 год.'],
 ['Винсент Ван Гог.', 'Эрмитаж.'],
 ['Уильям Шекспир.', 'Верона.'],
 ['Хью Джекман.', 'Девять.'],
 ['Наполеон I.', '1870 год.'],
 ['Бизе.', 'Кармен.'],
 ['Гюстав Эйфель и его сотрудники - Морис Кешелен и Эміль Нугьер.',
  '1889 год.'],
 ['Джосиф Сталин.', '1934 год.'],
 ['Альбертом Эйнштейном.', '1954 год.'],
 ['Несмотря на работу Антонио Меуччи, телефоном часто считают Александра Белла.',
  '1876 год.'],
 ['Лев Николаевич Толстой.', '1869 год.'],
 ['Леонардо да Винчи.', 'Санта-Мария-делле-Грацие.'],
 ['Билл Гейтс и Пол Аллен.', '1997 год.'],
 ['Хеат Леджер.',
  'Вопрос неверно сформулирован. Окончательный ответ неизвестен.'],
 ['Человеку не удалось первой посетить Землю с помощью к

In [ ]:
with open("/content/drive/My Drive/Colab Notebooks/DL/Трансформеры/karpov/LLM/Project/dialog_answers.json", "w") as f:
    json.dump(answers, f, ensure_ascii=False, indent=4)
print("Ответы сохранены в файл dialog_answers.json")